In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, lit
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

# not required
# spark

df = spark.read.option("header", "true")\
            .csv("/home/iceberg/data/events.csv")\
            .withColumn("event_date", expr("DATE_TRUNC('day', event_time)"))

df.show()
# dont evert run collect() on entire dataset as it will OOM and crash kernel, unless data is aggregated
# df.collect()

# this will oom and crash kernel
# df.join(df, lit(1) == lit(1)).collect()

# this ran as there was limited data
# df.join(df, lit(1) == lit(1)).take(5)

+-----------+----------+--------+--------------------+----------+--------------------+-------------------+
|    user_id| device_id|referrer|                host|       url|          event_time|         event_date|
+-----------+----------+--------+--------------------+----------+--------------------+-------------------+
| 1037710827| 532630305|    NULL| www.zachwilson.tech|         /|2021-03-08 17:27:...|2021-03-08 00:00:00|
|  925588856| 532630305|    NULL|    www.eczachly.com|         /|2021-05-10 11:26:...|2021-05-10 00:00:00|
|-1180485268| 532630305|    NULL|admin.zachwilson....|         /|2021-02-17 16:19:...|2021-02-17 00:00:00|
|-1044833855| 532630305|    NULL| www.zachwilson.tech|         /|2021-09-24 15:53:...|2021-09-24 00:00:00|
|  747494706| 532630305|    NULL| www.zachwilson.tech|         /|2021-09-26 16:03:...|2021-09-26 00:00:00|
|  747494706| 532630305|    NULL|admin.zachwilson....|         /|2021-02-21 16:08:...|2021-02-21 00:00:00|
| -824540328| 532630305|    NULL|admi

In [3]:
# sortWithinPartitions - sorts within a partition, recommended 
sorted = df.repartition(10, col("event_date"))\
    .sortWithinPartitions(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

# sort - does global sort across all data, not recommended
sortedTwo = df.repartition(10, col("event_date"))\
    .sort(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sorted.show()
sortedTwo.show()

+-----------+-----------+--------------------+--------------------+--------------------+--------------------+-------------------+
|    user_id|  device_id|            referrer|                host|                 url|          event_time|         event_date|
+-----------+-----------+--------------------+--------------------+--------------------+--------------------+-------------------+
| 1129583063|  532630305|                NULL|admin.zachwilson....|                   /|2021-01-07 09:21:...|2021-01-07 00:00:00|
| -648945006| 1088283544|                NULL|    www.eczachly.com|                   /|2021-01-07 02:58:...|2021-01-07 00:00:00|
|-1871780024| -158310583|                NULL|    www.eczachly.com|                   /|2021-01-07 04:17:...|2021-01-07 00:00:00|
|  203689086| 1088283544|                NULL|    www.eczachly.com|/blog/what-exactl...|2021-01-07 10:03:...|2021-01-07 00:00:00|
|-1180485268|  532630305|                NULL|    www.eczachly.com|                   /|20

In [ ]:
# .sortWithinPartitions() sorts within partitions, whereas .sort() is a global sort, which is very slow

# Note - exchange is synonymous with Shuffle

# to undestand explain plan, read from bottom to top
# FileScan - read csv file
# Project - selects the column
# Exchange - means shuffles ~ repartition line
# Sort - sorts within partition or global - check true/false flag in explain plan
# Project - final select 

# true means GLOBAL sort enabled in second plan. Also, it will only work with one executor doing a global sort
#   +- Sort [event_date#1835 ASC NULLS FIRST, host#1826 ASC NULLS FIRST], true, 0
# THIS `Exchange rangepartitioning` will be painful at scale - it will cause shuffle again after previous shuffle due to repartition
#      +- Exchange rangepartitioning(event_date#1835 ASC NULLS FIRST, host#1826 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=2695]

In [32]:
sorted = df.repartition(10, col("event_date"))\
    .sortWithinPartitions(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sortedTwo = df.repartition(10, col("event_date"))\
    .sort(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sorted.explain()
sortedTwo.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [device_id#266, user_id#265, referrer#267, host#268, url#269, cast(event_time#270 as timestamp) AS event_time#1310, event_date#277, browser_family#322, os_family#323, device_type#306]
   +- Sort [event_date#277 ASC NULLS FIRST, host#268 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(event_date#277, 10), REPARTITION_BY_NUM, [plan_id=1661]
         +- Project [device_id#266, user_id#265, referrer#267, host#268, url#269, event_time#270, event_date#277, browser_type#304 AS browser_family#322, os_type#305 AS os_family#323, device_type#306]
            +- BroadcastHashJoin [device_id#266], [device_id#303], LeftOuter, BuildRight, false
               :- Project [user_id#265, device_id#266, referrer#267, host#268, url#269, event_time#270, date_trunc(day, cast(event_time#270 as timestamp), Some(Etc/UTC)) AS event_date#277]
               :  +- FileScan csv [user_id#265,device_id#266,referrer#267,host#268,url#269,e

In [33]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, lit
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

events = spark.read.option("header", "true")\
            .csv("/home/iceberg/data/events.csv")\
            .withColumn("event_date", expr("DATE_TRUNC('day', event_time)"))

devices = spark.read.option("header","true")\
            .csv("/home/iceberg/data/devices.csv")

df = events.join(devices,on="device_id",how="left")
df = df.withColumnsRenamed({'browser_type': 'browser_family', 'os_type': 'os_family'})

df.show()

+----------+-----------+--------+--------------------+----------+--------------------+-------------------+--------------+---------+-----------+
| device_id|    user_id|referrer|                host|       url|          event_time|         event_date|browser_family|os_family|device_type|
+----------+-----------+--------+--------------------+----------+--------------------+-------------------+--------------+---------+-----------+
| 532630305| 1037710827|    NULL| www.zachwilson.tech|         /|2021-03-08 17:27:...|2021-03-08 00:00:00|         Other|    Other|      Other|
| 532630305|  925588856|    NULL|    www.eczachly.com|         /|2021-05-10 11:26:...|2021-05-10 00:00:00|         Other|    Other|      Other|
| 532630305|-1180485268|    NULL|admin.zachwilson....|         /|2021-02-17 16:19:...|2021-02-17 00:00:00|         Other|    Other|      Other|
| 532630305|-1044833855|    NULL| www.zachwilson.tech|         /|2021-09-24 15:53:...|2021-09-24 00:00:00|         Other|    Other|     

In [34]:
%%sql

CREATE DATABASE IF NOT EXISTS bootcamp

++
||
++
++

In [35]:
%%sql

DROP TABLE IF EXISTS bootcamp.events

++
||
++
++

In [36]:
%%sql

DROP TABLE IF EXISTS bootcamp.events_sorted

++
||
++
++

In [39]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.events (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (event_date);


++
||
++
++

In [40]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_sorted (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (event_date);

++
||
++
++

In [41]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_unsorted (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (event_date);

++
||
++
++

In [52]:

start_df = df.repartition(4, col("event_date")).withColumn("event_time", col("event_time").cast("timestamp")) \
    
first_sort_df = start_df.sortWithinPartitions(col("event_date"), col("host"))

start_df.write.mode("overwrite").saveAsTable("bootcamp.events_unsorted")
first_sort_df.write.mode("overwrite").saveAsTable("bootcamp.events_sorted")

In [53]:
%%sql

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted' 
FROM demo.bootcamp.events_sorted.files

UNION ALL
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'unsorted' 
FROM demo.bootcamp.events_unsorted.files





size,num_files,sorted
5444946,4,sorted
5556672,4,unsorted


In [44]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files FROM demo.bootcamp.events.files;

size,num_files
None,0


In [45]:
%%sql 
SELECT COUNT(1) FROM bootcamp.matches_bucketed.files

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `bootcamp`.`matches_bucketed`.`files` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 21;
'Aggregate [unresolvedalias(count(1), None)]
+- 'UnresolvedRelation [bootcamp, matches_bucketed, files], [], false


In [46]:
%%sql

SELECT *
FROM demo.bootcamp.events_sorted.files

content,file_path,file_format,spec_id,partition,record_count,file_size_in_bytes,column_sizes,value_counts,null_value_counts,nan_value_counts,lower_bounds,upper_bounds,key_metadata,split_offsets,equality_ids,sort_order_id,referenced_data_file,content_offset,content_size_in_bytes,readable_metrics
0,s3://warehouse/bootcamp/events_sorted/data/00000-159-8e8be00f-b845-451f-b45a-031e97f06779-0-00001.parquet,PARQUET,1,Row(event_date=None),89391,1032261,"{1: 107466, 2: 61022, 3: 11455, 4: 12926, 6: 7383, 7: 426449, 8: 2293, 9: 77425, 10: 310063, 11: 10711}","{1: 89391, 2: 89391, 3: 89391, 4: 89391, 6: 89391, 7: 89391, 8: 89391, 9: 89391, 10: 89391, 11: 89391}","{1: 0, 2: 46359, 3: 0, 4: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 1, 11: 0}",{},"{1: bytearray(b'/'), 2: bytearray(b'52.20.78.240'), 3: bytearray(b'%E3%82%A6%E3%82%'), 4: bytearray(b'Android'), 6: bytearray(b'aashish.techcrea'), 7: bytearray(b' \xba\xe7\xb8\xa8\xb8\x05\x00'), 8: bytearray(b'\x00\xa0&\xb4\xa8\xb8\x05\x00'), 9: bytearray(b'-100210680'), 10: bytearray(b'-1000095488'), 11: bytearray(b'17MB150WB')}","{1: bytearray(b'/zzageqnf.php?Fp'), 2: bytearray(b'zachwilson.tech'), 3: bytearray(b'webprosbot'), 4: bytearray(b'iOS'), 6: bytearray(b'zachwilson.techd'), 7: bytearray(b'\xe8\xb0\x1b\x8ec\x03\x06\x00'), 8: bytearray(b'\x00\xe0dqO\x03\x06\x00'), 9: bytearray(b'999535123'), 10: bytearray(b'999884938'), 11: bytearray(b'vivo $2')}",None,[4],None,0,None,None,None,"Row(browser_family=Row(column_size=11455, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='%E3%82%A6%E3%82%', upper_bound='webprosbot'), device_id=Row(column_size=77425, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='-100210680', upper_bound='999535123'), device_type=Row(column_size=10711, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='17MB150WB', upper_bound='vivo $2'), event_date=Row(column_size=2293, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound=datetime.datetime(2021, 1, 12, 0, 0), upper_bound=datetime.datetime(2023, 8, 20, 0, 0)), event_time=Row(column_size=426449, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound=datetime.datetime(2021, 1, 12, 0, 1, 19, 764000), upper_bound=datetime.datetime(2023, 8, 20, 23, 59, 41, 89000)), host=Row(column_size=7383, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='aashish.techcrea', upper_bound='zachwilson.techd'), os_family=Row(column_size=12926, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='Android', upper_bound='iOS'), referrer=Row(column_size=61022, value_count=89391, null_value_count=46359, nan_value_count=None, lower_bound='52.20.78.240', upper_bound='zachwilson.tech'), url=Row(column_size=107466, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='/', upper_bound='/zzageqnf.php?Fp'), user_id=Row(column_size=310063, value_count=89391, null_value_count=1, nan_value_count=None, lower_bound='-1000095488', upper_bound='999884938'))"
0,s3://warehouse/bootcamp/events_sorted/data/00001-160-8e8be00f-b845-451f-b45a-031e97f06779-0-00001.parquet,PARQUET,1,Row(event_date=None),99232,1165546,"{1: 142178, 2: 67363, 3: 11914, 4: 16543, 6: 9119, 7: 475862, 8: 2373, 9: 86514, 10: 337013, 11: 11522}","{1: 99232, 2: 99232, 3: 99232, 4: 99232, 6: 99232, 7: 99232, 8: 99232, 9: 99232, 10: 99232, 11: 99232}","{1: 0, 2: 49299, 3: 0, 4: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 58, 11: 0}",{},"{1: bytearray(b'""/?""""<?=print(93'), 2: bytearray(b'""https://www.goo'), 3: bytearray(b') Bot'), 4: bytearray(b'Android'), 6: bytearray(b'abhishekanand.te'), 7: bytearray(b'(\x83\xb2EX\xb8\x05\x00'), 8: bytearray(b'\x00 \xc9<X\xb8\x05\x00'), 9: bytearray(b'-100210680'), 10: bytearray(b'-1000370060'), 11: bytearray(b'13 Pro Max')}","{1: bytearray(b'/zz.php'), 2: bytearray(b'zachwilson.tech'), 3: bytearray(b'webprosbot'), 4: bytearray(b'iOS'), 6: bytearray(b'zsavi524.techcrf'), 7: bytearray(b'\x88\xb8\x07P;\x03\x06\x00'), 

In [23]:
%%sql

SELECT *
FROM demo.bootcamp.events_sorted

device_id,user_id,referrer,host,url,event_time,event_date,browser_family,os_family,device_type
532630305,-488618451,None,admin.zachwilson.tech,/,2021-01-12 10:22:17.016000,2021-01-12 00:00:00,Other,Other,Other
589185851,-21136712,None,admin.zachwilson.tech,/,2021-01-12 18:49:28.425000,2021-01-12 00:00:00,Chrome,Linux,Other
589185851,-414920062,None,admin.zachwilson.tech,/,2021-01-12 18:54:30.995000,2021-01-12 00:00:00,Chrome,Linux,Other
589185851,-414920062,None,admin.zachwilson.tech,/,2021-01-12 19:56:56.809000,2021-01-12 00:00:00,Chrome,Linux,Other
589185851,-694958230,None,admin.zachwilson.tech,/,2021-01-12 20:08:15.964000,2021-01-12 00:00:00,Chrome,Linux,Other
-290659081,2105351485,None,www.eczachly.com,/,2021-01-12 04:44:23.791000,2021-01-12 00:00:00,bingbot,Other,Spider
-843023486,-2116612468,None,www.eczachly.com,/,2021-01-12 01:13:53.762000,2021-01-12 00:00:00,Chrome,Mac OS X,Other
-843023486,-2116612468,https://www.eczachly.com/,www.eczachly.com,/blog,2021-01-12 01:13:56.914000,2021-01-12 00:00:00,Chrome,Mac OS X,Other
-843023486,-2116612468,https://www.eczachly.com/blog,www.eczachly.com,/graphs,2021-01-12 01:13:58.899000,2021-01-12 00:00:00,Chrome,Mac OS X,Other
-843023486,-2116612468,https://www.eczachly.com/graphs,www.eczachly.com,/graphs,2021-01-12 01:14:01.017000,2021-01-12 00:00:00,Chrome,Mac OS X,Other
